In [1]:
import os
from pathlib import Path
import torch
import torch.nn as nn 
from torch.nn import functional as F
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [2]:

# hyperparameters
max_iters = 100
eval_interval = 200
block_size = 8
batch_size = 4
learning_rate = 1e-2
n_emb = 32


In [3]:

def get_data():
    # data file path
    shakespeare_txt = Path(r"../data/raw/shakespeare.txt")

    if not os.path.exists(shakespeare_txt):
        raise FileNotFoundError(f"File not found: {shakespeare_txt}")

    if shakespeare_txt.exists():
        logger.info(f"file found : {shakespeare_txt.name}")
        logger.info(f"loading file : `{shakespeare_txt.name}` of size : {shakespeare_txt.stat().st_size / (1024)} KB")

        text = shakespeare_txt.read_text()
        logger.info(f"length of dataset in character : {len(text)}")
        logger.info(f"sample text : \n {text[:50]}")
    return text

# get the data
text = get_data()

# sorted list of unique characters in data
char = sorted(list(set(text)))

# vocabulary size
vocab_size = len(char)

logger.info(f"vocab_size : {vocab_size}")
logger.info(f"unique characters : {str().join(char)}")

# create encoder and decoder for character level
stoi = {ch:i for i, ch in enumerate(char)}
itos = {i:ch for i, ch in enumerate(char)}

encoder = lambda s : [stoi[c] for c in s]
decoder = lambda l : "".join([itos[i] for i in l])

# create train and test dataset
data = torch.tensor(encoder(text), dtype=torch.long)

n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x,y

2025-12-10 15:22:38,127 - __main__ - INFO - file found : shakespeare.txt
2025-12-10 15:22:38,129 - __main__ - INFO - loading file : `shakespeare.txt` of size : 306.9033203125 KB
2025-12-10 15:22:38,138 - __main__ - INFO - length of dataset in character : 306996
2025-12-10 15:22:38,141 - __main__ - INFO - sample text : 
 O for a Muse of fire, that would ascend
The bright
2025-12-10 15:22:38,146 - __main__ - INFO - vocab_size : 79
2025-12-10 15:22:38,147 - __main__ - INFO - unique characters : 
 !"'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz


### Mathematical trick in self-attention

In [5]:
# torch.manual_seed(1337)
B,T,C = 4,8,2

x = torch.randn(B,T,C)


#### version 1

In [6]:

# bag of words
xbow = torch.zeros((B,T,C))

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b,t] = torch.mean(xprev, dim=0)

x[0], xbow[0]

(tensor([[-0.5295,  0.6730],
         [-0.9692,  0.0275],
         [ 0.5366,  0.1969],
         [-2.3294, -2.1396],
         [-0.6339, -1.7413],
         [ 1.0991,  0.5940],
         [ 1.2871, -0.8275],
         [ 1.0846,  0.7060]]),
 tensor([[-0.5295,  0.6730],
         [-0.7493,  0.3502],
         [-0.3207,  0.2991],
         [-0.8229, -0.3106],
         [-0.7851, -0.5967],
         [-0.4711, -0.3983],
         [-0.2199, -0.4596],
         [-0.0568, -0.3139]]))

#### version 2

In [7]:
# torch.one() : creates tensor of one
# torch.tril() : converts tensor to lower digonal matrix 
wei = torch.tril(torch.ones(T,T))
wei = wei/wei.sum(-1, keepdim=True)

# (T,T) @ (B, T, C) ----> (B, T, C)
xbow2 =  wei @ x

torch.allclose(xbow, xbow2)

True

#### version 3

In [8]:
# lower triangular matrix
tril = torch.tril(torch.ones(T,T))

wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float("-inf"))
mask = wei.softmax(dim=-1)

xbow3 = mask @ x

torch.allclose(xbow, xbow3)

True

In [9]:
xbow[0], xbow2[0], xbow3[0]

(tensor([[-0.5295,  0.6730],
         [-0.7493,  0.3502],
         [-0.3207,  0.2991],
         [-0.8229, -0.3106],
         [-0.7851, -0.5967],
         [-0.4711, -0.3983],
         [-0.2199, -0.4596],
         [-0.0568, -0.3139]]),
 tensor([[-0.5295,  0.6730],
         [-0.7493,  0.3502],
         [-0.3207,  0.2991],
         [-0.8229, -0.3106],
         [-0.7851, -0.5967],
         [-0.4711, -0.3983],
         [-0.2199, -0.4596],
         [-0.0568, -0.3139]]),
 tensor([[-0.5295,  0.6730],
         [-0.7493,  0.3502],
         [-0.3207,  0.2991],
         [-0.8229, -0.3106],
         [-0.7851, -0.5967],
         [-0.4711, -0.3983],
         [-0.2199, -0.4596],
         [-0.0568, -0.3139]]))

#### version 4 - self attention

In [48]:
import math 

x = torch.randn(B,T,C)
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

tril = torch.tril(torch.ones(T, T))

k = key(x)
q = key(x)
v = key(x)

wei = q @ k.transpose(-2, -1) / math.sqrt(head_size)
wei = wei.masked_fill(tril == 0 , float("-inf"))
wei = wei.softmax(-1)
wei.shape

attention = wei @ v
attention.shape

torch.Size([4, 8, 16])

#### Train loop

In [67]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_interval)
        for k in range(eval_interval):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def train(model, optimizer, max_iters=max_iters):
    for iter in range(max_iters):

        # every once in a while evaluate the loss on train and val sets
        if iter % eval_interval == 0:
            losses = estimate_loss(model)
            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # get the batch
        xb, yb = get_batch('train')

        # evalute loss
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True) # make the previous gradients to none
        loss.backward() # calculate new gradients
        optimizer.step() # update the parameters

## Language Model - 1

In [22]:
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T) 
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        x = tok_emb + pos_emb  # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, -1)
            target = target.view(-1)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        for _ in range(max_tokens):
            
            # get predictions
            logits, loss = self(idx[: , -block_size:])

            # get the last word/character predictions
            logits = logits[:, -1, :] # (B, 1, C)

            # get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # get the next token id by sampling from the distribution
            id = torch.multinomial(probs, num_samples=1) # (B, 1) <=> (1,1)

            # append sampled index to the running sequence
            idx = torch.cat((idx, id), dim=-1)
        return idx

In [23]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [25]:
train(model, optimizer, 2000)

step 0: train loss 4.7527, val loss 4.7758
step 200: train loss 2.6892, val loss 2.8386
step 400: train loss 2.5905, val loss 2.7237
step 600: train loss 2.5773, val loss 2.6590
step 800: train loss 2.6301, val loss 2.6854
step 1000: train loss 2.5874, val loss 2.6954
step 1200: train loss 2.5604, val loss 2.6588
step 1400: train loss 2.5090, val loss 2.6276
step 1600: train loss 2.5406, val loss 2.5958
step 1800: train loss 2.5148, val loss 2.6488


In [31]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))


 the  r ty gidoulets,
 t?12Lisas seshan!  tet   a  jode.  The med,
 ploun ul'drouqul;
 ESowhy DENofreld AThereses Burticalo he Thouremet al I m snorureersk myed arayo IBEpuns bllofulel s tatase, fatofoded te alsouved s;
 wrd Ivillforere
SLOUSS. wigouse prilemis, fasurgind ane 5151
 knon  f whe. yory,

 allancot te.  masele g aton mpont  prerred,
 illas merelfrssithakeeaallesril  oud surd l'd. tmelr, terer s ts,
 LONin
 myatinddsingaioture,
 ns aevert; winosereererims G ll,
  ind  ROmasis  ano  t


## Language Model - 2

- Added self attention

B = batch, 
T = block_size/context_length/time,  
C = channel/embedding_length/embd_dim
H = head_size

x => B,T,C

embedding_table => vocabulary , embd_dim
positional_table => block_size, embd_dim

x = embedding_table(x) + positional_table(x)  ===> B, T, C ===> batch, block_size, embd_dim

wk, wq, wv ===> (embd_dim, H)

k = wk(x)  ===> B , T, H
q = wq(x)  ===> B , T, H
v = wv(x)  ===> B , T, H

w = q @ k.T  ===> (B , T, H) @ (B , H, T) = (B , T, T)
w  = softmax(w)  ===> (B , T, T)

attention = w @ v  ===> (B , T, T) @ (B , T, H) = (B, T, H)
ff ===> 

In [125]:
# from dataclasses import dataclass
vocab_size: int = vocab_size
batch_size: int = 32  # number of independent examples
block_size: int = 16  # maximum tokens handled by llm at a time (context length)
learning_rate: float = 1e-3
n_emb = 32
eval_iters = 200

In [135]:

import math 
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)

        k_dim = k.shape[-1]
        # compute attention score
        wei = q @ k.transpose(-2, -1) / math.sqrt(k_dim)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        wei = wei.softmax(-1)  # (B, T, T)
        
        out = wei @ v # (B, T, H)
        return out

class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.sa_head = Head(n_emb)
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        x = tok_emb + pos_emb  # (B, T, C)
        
        x = self.sa_head(x)
        
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        # idx is (B, T) array of indices in the current context
        for _ in range(max_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [136]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [127]:
train(model=model, optimizer=optimizer, max_iters=5000)

step 0: train loss 4.3844, val loss 4.3812
step 200: train loss 3.0431, val loss 3.0952
step 400: train loss 2.8250, val loss 2.8641
step 600: train loss 2.6764, val loss 2.7213
step 800: train loss 2.5506, val loss 2.5949
step 1000: train loss 2.4970, val loss 2.5484
step 1200: train loss 2.4657, val loss 2.5175
step 1400: train loss 2.4425, val loss 2.5106
step 1600: train loss 2.4226, val loss 2.4922
step 1800: train loss 2.4143, val loss 2.4975
step 2000: train loss 2.3988, val loss 2.4744
step 2200: train loss 2.3982, val loss 2.4656
step 2400: train loss 2.3932, val loss 2.4586
step 2600: train loss 2.3808, val loss 2.4522
step 2800: train loss 2.3777, val loss 2.4565
step 3000: train loss 2.3726, val loss 2.4488
step 3200: train loss 2.3593, val loss 2.4422
step 3400: train loss 2.3634, val loss 2.4355
step 3600: train loss 2.3542, val loss 2.4301
step 3800: train loss 2.3518, val loss 2.4289
step 4000: train loss 2.3583, val loss 2.4180
step 4200: train loss 2.3450, val loss 2.

In [128]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))


     lolowloust catsorukecra r' fithe, me ithtlilfo in'he, s sblof me se ofa t heinou  ser,
    T y med stalmy t save.  orcis t when mome,
  N  inh be t owla ma t stto-we:

     A.  toyad s th hord w e ewcher, a bali'th Namy tounsenss, I  g on, owntcatthans,
 Nord, wem, bers wartu yotoen: ss heu  t phann courou thito,
     tave borces ma
    thin?
  ICOPU-ra xad.
    frout  chis o s t thin sythet myel cese, t  skshe,
    Th' thunang t oueri m wagher tldavengaincy
    D  ms benpoou cckng t; d cen


## Language Model - 3

Added Multihead attention 

`MultiHeadAttention(4, n_emb//4)`

- number of heads (num_heads = 4)
  

In [120]:
# hyperparameters
vocab_size: int = vocab_size
batch_size: int = 32  # number of independent examples
block_size: int = 8  # maximum tokens handled by llm at a time (context length)
learning_rate: float = 1e-3
n_emb = 32
eval_iters = 200

In [143]:

import math 
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)

        k_dim = k.shape[-1]
        # compute attention score
        wei = q @ k.transpose(-2, -1) / math.sqrt(k_dim)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        wei = wei.softmax(-1)  # (B, T, T)
        
        out = wei @ v # (B, T, H)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.sa_head = MultiHeadAttention(4, n_emb//4)
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        x = tok_emb + pos_emb  # (B, T, C)
        
        x = self.sa_head(x)
        
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        # idx is (B, T) array of indices in the current context
        for _ in range(max_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [144]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [145]:
train(model=model, optimizer=optimizer, max_iters=5000)

step 0: train loss 4.4305, val loss 4.4286
step 200: train loss 2.9382, val loss 2.9909
step 400: train loss 2.7145, val loss 2.7617
step 600: train loss 2.6280, val loss 2.6719
step 800: train loss 2.5680, val loss 2.6205
step 1000: train loss 2.5184, val loss 2.5862
step 1200: train loss 2.4922, val loss 2.5490
step 1400: train loss 2.4494, val loss 2.5151
step 1600: train loss 2.4079, val loss 2.4731
step 1800: train loss 2.3788, val loss 2.4395
step 2000: train loss 2.3513, val loss 2.4234
step 2200: train loss 2.3219, val loss 2.4005
step 2400: train loss 2.2989, val loss 2.3684
step 2600: train loss 2.2855, val loss 2.3570
step 2800: train loss 2.2751, val loss 2.3388
step 3000: train loss 2.2412, val loss 2.3140
step 3200: train loss 2.2179, val loss 2.3144
step 3400: train loss 2.2041, val loss 2.3027
step 3600: train loss 2.1865, val loss 2.2880
step 3800: train loss 2.1807, val loss 2.2866
step 4000: train loss 2.1702, val loss 2.2760
step 4200: train loss 2.1688, val loss 2.

In [146]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))


 The  o sto  ther, Do dout fascm dme, I goveirm colestr will,
  PEXICTRETESEN. Fow ald. Fovem thith ou cinge ealll thiceks cie.
  Whe thee min, of cut nivem wis thais wey wired nou inge, allive erraicllfighoull.
  Thaflet cont a thusuts ses fangn:
  MINGE, A Chrave nyou cowl sexcend, mine pane' sintorn,
  Cire thar thit fore, wee ancomelol for,
  'The chen, avith's an tha suskfreit hie wie hy onon ow.
     Mion, thes muct orw pupss sew oady to aing?
 sd LOUS!
  Gees hireal'c.

  For'sesstrem ane


## Language Model - 4
In this language model is `FeedForward layer` is added.

Now  

$Model = embedding + position_embedding + (MultiheadAttention + FeedForward)$

$FeedForward = LinearLayer + Relu$

In [148]:
# hyperparameters
vocab_size: int = vocab_size
batch_size: int = 32  # number of independent examples
block_size: int = 8  # maximum tokens handled by llm at a time (context length)
learning_rate: float = 1e-3
n_emb = 32
eval_iters = 200

In [147]:
import math 

class FeedForward(nn.Module):
    """ a simple feedforward layer of Linear layer followed by non-linearity. It is also called as MultiLayer Perceptron (MLP)."""

    def __init__(self, n_emb):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_emb, n_emb),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)

        k_dim = k.shape[-1]
        # compute attention score
        wei = q @ k.transpose(-2, -1) / math.sqrt(k_dim)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        wei = wei.softmax(-1)  # (B, T, T)
        
        out = wei @ v # (B, T, H)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return out

class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.sa_head = MultiHeadAttention(4, n_emb//4)
        self.ff = FeedForward(n_emb)
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        # adding token embedding and position embeding
        x = tok_emb + pos_emb  # (B, T, C)
        
        # applying multihead attention layer
        x = self.sa_head(x)
        
        # applying feed forward layer
        x = self.ff(x)

        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        # idx is (B, T) array of indices in the current context
        for _ in range(max_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [149]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [150]:
train(model=model, optimizer=optimizer, max_iters=5000)

step 0: train loss 4.3750, val loss 4.3697
step 200: train loss 2.9624, val loss 3.0074
step 400: train loss 2.6806, val loss 2.7113
step 600: train loss 2.5119, val loss 2.5676
step 800: train loss 2.4219, val loss 2.4974
step 1000: train loss 2.3812, val loss 2.4569
step 1200: train loss 2.3558, val loss 2.4132
step 1400: train loss 2.3149, val loss 2.3993
step 1600: train loss 2.3130, val loss 2.3849
step 1800: train loss 2.2707, val loss 2.3703
step 2000: train loss 2.2634, val loss 2.3431
step 2200: train loss 2.2449, val loss 2.3343
step 2400: train loss 2.2400, val loss 2.3169
step 2600: train loss 2.2209, val loss 2.3180
step 2800: train loss 2.2106, val loss 2.3199
step 3000: train loss 2.1957, val loss 2.2942
step 3200: train loss 2.1928, val loss 2.2954
step 3400: train loss 2.1853, val loss 2.2825
step 3600: train loss 2.1756, val loss 2.2877
step 3800: train loss 2.1593, val loss 2.2631
step 4000: train loss 2.1502, val loss 2.2560
step 4200: train loss 2.1490, val loss 2.

In [151]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))



    Hyus, you Thy histere my's food tots blillomnowesel the his, I me,
   To 's on, welth hur thichbis Rorenye ad My pere mony do kinde; thath my butue, pinve
   LONG Exind,
    To thin s pin sust hour tut let mad, in ths whome st ancut,
  Has and, mas;
  WISTES UONTECONDS. Whor sum theno trund,
E  The by hour hit peat bee deand cosit his savicle gurer loot maind hour,
 
  'As
  Mat fur the wlelys that thores by? Offove, utuer furifirn the exp a ad hered ininge, ajequnee vere,
              Aty


## Language Model - 5
- In the Model Multiple Transformer Blocks are added.
- each Transformer Block contains MultiHeadAttention Layer and FeedForward Layer.  
- In each block `residual connection` is added which is also called as `skip connections`. 
  - they help vanishing gradient problem and used as regularization technique.
- `MultiHeadAttention` Block : contains Multihead Attention followed by Linear Projection Layer.
- `FeedForward` layer : conatains Linear Layer followed by Relu Non-linearity and Linear Projection Layer.
  - In FeedForward layer first Linear layer is expanded to size *(n_emb, n_emb\*2)* and again compressed back to *(n_emb\*2, n_emb)*
  

> `Important` : In GPT paper for feed-forward layer, the compression ratio is of 4 for linear layer. i.e 1st linear later is *(n_emb, n_emb\*4)* and last linear layer is *(n_emb\*4, n_emb)*, But we used compression ratio as 2.


In [152]:
import math 

class FeedForward(nn.Module):
    """ a simple feedforward layer of Linear layer followed by non-linearity. It is also called as MultiLayer Perceptron (MLP)."""

    def __init__(self, n_emb):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_emb, n_emb*2),
            nn.ReLU(),
            nn.Linear(n_emb*2, n_emb)
        )

    def forward(self, x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)

        k_dim = k.shape[-1]
        # compute attention score
        wei = q @ k.transpose(-2, -1) / math.sqrt(k_dim)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        wei = wei.softmax(-1)  # (B, T, T)
        
        out = wei @ v # (B, T, H)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_emb, n_emb)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out


class Block(nn.Module):
    """ Transformer Block: Self Attention + FeedForward Layer with residual connection"""

    def __init__(self, n_emb, n_heads):
        super().__init__()
        head_size = n_emb//n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward(n_emb)

    def forward(self, x):
        x = x + self.sa(x)
        x = x + self.ff(x)
        return x

class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.block = nn.Sequential(
            Block(n_emb, n_heads=4),
            Block(n_emb, n_heads=4),
            Block(n_emb, n_heads=4),
        )
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        # adding token embedding and position embeding
        x = tok_emb + pos_emb  # (B, T, C)
        
        # applying transfomer blocks
        x = self.block(x)

        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        # idx is (B, T) array of indices in the current context
        for _ in range(max_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [153]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [154]:
train(model=model, optimizer=optimizer, max_iters=5000)

step 0: train loss 4.9039, val loss 4.9185
step 200: train loss 2.5553, val loss 2.6121
step 400: train loss 2.3586, val loss 2.4546
step 600: train loss 2.2911, val loss 2.3808
step 800: train loss 2.2269, val loss 2.3245
step 1000: train loss 2.2031, val loss 2.2842
step 1200: train loss 2.1487, val loss 2.2541
step 1400: train loss 2.1257, val loss 2.2508
step 1600: train loss 2.0809, val loss 2.2422
step 1800: train loss 2.0866, val loss 2.2192
step 2000: train loss 2.0833, val loss 2.1747
step 2200: train loss 2.0574, val loss 2.1827
step 2400: train loss 2.0329, val loss 2.1625
step 2600: train loss 2.0141, val loss 2.1478
step 2800: train loss 2.0314, val loss 2.1702
step 3000: train loss 2.0096, val loss 2.1170
step 3200: train loss 1.9799, val loss 2.1202
step 3400: train loss 1.9867, val loss 2.1368
step 3600: train loss 1.9643, val loss 2.1026
step 3800: train loss 1.9670, val loss 2.1209
step 4000: train loss 1.9579, val loss 2.0986
step 4200: train loss 1.9458, val loss 2.

In [155]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))


  BENANTESS. Thow well,
    Bet red that the baisserving stirs'sy.
  PAROLLLORD That Cathone,
Fut
  Eve woded; loved forse to refornd beatty. And ay be whist the hath fing one hose:
    Whon on's in to allienn of that but to firg, arun'd fore, honavel.
  HHAF. TIT DIAM is, afllord, bodself it that in upondant beasest,
  But now, I fray may tied not, nots hands;
  The'sing.
    Long mo] Sir; had' I morive sure things,
  LAFEU. ROICTHS
  CHELENES. Longt.
  DTENTESS. Yet suceh bo, repessountrainsal


## Language Model - 6

In this version, Layer Normalisation is added as a regularisation techinique which helps in vanishig/exploading gradient descent problem.  
Layer norm is applied before it is going to MultiHeadAttention and FeedForward layer

The formula for Layer Normalization is:

$$y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$
  
Here, $\sigma$ is std. deviation

* $x$: The **input vector** to the layer for a single training example.
* $\mu$: The **Mean** of the input vector, calculated across all the features of a single example.
* standard deviation is the square root of the variance (${SD}(\mathbf{x}) = \sqrt{variance} =\sqrt{\sigma^2} = \sigma $)
* $\sigma^2$: The **Variance** of the input vector, also calculated across the features of a single example. (${Var}(\mathbf{x}) = \frac{1}{n} \sum_{j=1}^n (x_j - \mu(\mathbf{x}))^2$)

* $\epsilon$: A small constant added for **numerical stability** to prevent division by zero.
* $\gamma$: A learnable **scaling parameter** (also known as a gain).
* $\beta$: A learnable **shifting parameter** (also known as a bias).

The learnable parameters, $\gamma$ and $\beta$, enable the model to learn the optimal scale and shift for the normalized output, which can be critical for the network's expressiveness and performance.


In [176]:
# hyperparameters
vocab_size: int = vocab_size
batch_size: int = 32  # number of independent examples
block_size: int = 64  # maximum tokens handled by llm at a time (context length)
learning_rate: float = 3e-4
n_emb = 128
n_head = 6
n_layer = 6
dropout = 0.2
max_iters = 5000
eval_iters = 500

In [177]:
import math 

class LayerNormalization(nn.Module):
    """ Layer Normalisation """
    def __init__(self, features, eps=1e-6):
        super().__init__()

        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))

    def forward(self, x: torch.Tensor):
        # x ==> (B, T, C)
        # (B = Batch Size), (T = `block_size`, or `context_length`), (C = Embedding Dimension 'emb_dim' or 'features')

        x_mean = x.mean(dim=-1, keepdim=True)  # mean across vector dimension ==> x.size(2)
        variance = x.var(-1, keepdim=True, unbiased=False)
        denominator = torch.sqrt(variance + self.eps) #<==> (x_std + self.eps)

        x_norm = (x - x_mean) / denominator
        x = self.gamma * x_norm + self.beta

        return x
    

class FeedForward(nn.Module):
    """ a simple feedforward layer of Linear layer followed by non-linearity. It is also called as MultiLayer Perceptron (MLP)."""

    def __init__(self, n_emb):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_emb, n_emb*2),
            nn.ReLU(),
            nn.Linear(n_emb*2, n_emb), 
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_emb, head_size, bias=False)
        self.query = nn.Linear(n_emb, head_size, bias=False)
        self.value = nn.Linear(n_emb, head_size, bias=False)        
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B, T, H)
        q = self.query(x)  # (B, T, H)
        v = self.value(x)  # (B, T, H)

        k_dim = k.shape[-1]
        # compute attention score
        wei = q @ k.transpose(-2, -1) / math.sqrt(k_dim)  # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))   # (B, T, T)
        wei = wei.softmax(-1)  # (B, T, T)
        wei = self.dropout(wei)
        
        out = wei @ v # (B, T, H)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_emb, n_emb)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class Block(nn.Module):
    """ Transformer Block: Self Attention + FeedForward Layer with residual connection"""

    def __init__(self, n_emb, n_heads):
        super().__init__()
        head_size = n_emb//n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ff = FeedForward(n_emb)
        self.ln1 = nn.LayerNorm(n_emb)
        self.ln2 = nn.LayerNorm(n_emb)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # a lookup table for inputs
        self.token_embedding_table = nn.Embedding(vocab_size, n_emb)
        self.pos_embedding_table = nn.Embedding(block_size, n_emb)
        self.block = nn.Sequential(
            Block(n_emb, n_heads=4),
            Block(n_emb, n_heads=4),
            Block(n_emb, n_heads=4),
            nn.LayerNorm(n_emb),
        )
        self.lm_head = nn.Linear(n_emb, vocab_size)

    def forward(self, idx, target=None):
        # idx :: batch_size(B)  *  block_size(T)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # B,T,C
        pos_emb = self.pos_embedding_table(torch.arange(T))  # (T, C)

        # adding token embedding and position embeding
        x = tok_emb + pos_emb  # (B, T, C)
        
        # applying transfomer blocks
        x = self.block(x)

        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits, target)

        return logits, loss

    def generate(self, idx, max_tokens = 50):
        # Assuming ==> idx :: (B, T) : (1, T) 
        # i.e  Only a batch with input ==>  batch_size(B) = 1
        # idx is (B, T) array of indices in the current context
        for _ in range(max_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [178]:
model = LanguageModel()

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [179]:
train(model=model, optimizer=optimizer, max_iters=5000)

step 0: train loss 4.4902, val loss 4.4893
step 200: train loss 2.5183, val loss 2.5996
step 400: train loss 2.3482, val loss 2.4293
step 600: train loss 2.2056, val loss 2.3006
step 800: train loss 2.1058, val loss 2.2196
step 1000: train loss 2.0372, val loss 2.1545
step 1200: train loss 1.9785, val loss 2.1099
step 1400: train loss 1.9228, val loss 2.0627
step 1600: train loss 1.8742, val loss 2.0306
step 1800: train loss 1.8418, val loss 2.0038
step 2000: train loss 1.8089, val loss 1.9741
step 2200: train loss 1.7893, val loss 1.9543
step 2400: train loss 1.7635, val loss 1.9302
step 2600: train loss 1.7347, val loss 1.9071
step 2800: train loss 1.7248, val loss 1.8952
step 3000: train loss 1.7019, val loss 1.8757
step 3200: train loss 1.6892, val loss 1.8664
step 3400: train loss 1.6688, val loss 1.8467
step 3600: train loss 1.6620, val loss 1.8295
step 3800: train loss 1.6532, val loss 1.8171
step 4000: train loss 1.6320, val loss 1.8049
step 4200: train loss 1.6257, val loss 1.

In [180]:
# generate a sequence
context = torch.zeros((1,1), dtype=torch.long)

print(decoder(model.generate(context, 500).view(-1).tolist()))


    delled plit thy aucrest to le'er pirseeds.
  SCESTEU. And Woold adder PARBU.
    Do, wout I ruble, now
  Herprow it at his thy she heart.
  KINEs will 
Net here aleaving.
    My the strufth me cath stagere as feae; him
     as arm;
    So sturrad's thi's ned it sao of theren follot
    that hearn bractiin?                               E'MAKENGERBERT


    Thou and the KING. Caupt, this your lead, to thrust theu be out;
His ais it For the nole, in undee.
  SCEDORD. Be own thumbst merie for t
